# 01 · Análisis exploratorio de datos (EDA)

Fase 2 de CRISP-DM: comprensión de los datos. Todo el análisis se hace **solo sobre el train limpio** (40.690 filas, sin las filas que estaban repetidas en test). El hold-out no se mira para no sesgar decisiones.

La lógica está en `src/bank_captacion/eda.py`; este notebook la ejecuta y comenta los resultados.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display, Markdown
from bank_captacion import config as cfg
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def show(fig):
    display(Image(filename=str(cfg.FIGURES_DIR / fig)))

def table(name, **kw):
    df = pd.read_csv(cfg.TABLES_DIR / name, **kw)
    display(df)
    return df


In [ ]:
from bank_captacion.eda import run_eda
key = run_eda()
print(json.dumps(key, indent=2, default=float))

## 1. Preparación previa del dataset (Fase 0)

Antes del EDA se verificó el esquema y se eliminó el solapamiento train/test. La tabla siguiente es el registro completo de pasos aplicados a las filas.

In [ ]:
table('data_preparation_steps.csv')
table('data_summary.csv')

La partición train/test es **aleatoria**: las filas de test están distribuidas uniformemente a lo largo del archivo (ordenado cronológicamente) y las distribuciones de mes, tipo de contacto, resultado previo y target no difieren (p > 0,05 en todos los tests).

In [ ]:
table('split_checks.csv')

## 2. Variable objetivo

Desbalance moderado: 11,7 % de conversiones. Un clasificador trivial que responde siempre "no" tendría 88,3 % de accuracy, por eso accuracy no se usa.

In [ ]:
show('eda_target_balance.png')

## 3. Variables numéricas

In [ ]:
show('eda_numeric_distributions.png')
table('eda_numeric_by_class.csv')

- **Edad**: los que convierten están sobrerrepresentados en los extremos (< 30 y > 60 años: estudiantes y jubilados). Relación no lineal → justifica `age_group` además de `age`.
- **Saldo**: muy asimétrico (asimetría ≈ 8,5), con negativos y ceros. Los que convierten tienen saldos algo mayores.
- **Contactos en la campaña**: más contactos se asocian a menor conversión (insistir no ayuda).
- **pdays**: entre los contactados antes, picos en ~90 y ~180 días con alta conversión (probablemente seguimientos trimestrales/semestrales).

## 4. Variables categóricas: tasa de conversión por categoría

In [ ]:
show('eda_conversion_by_category.png')
conv = table('eda_conversion_by_category.csv')

- **poutcome = success**: 64,8 % de conversión (5,5 veces la base). Es la señal más fuerte disponible antes de llamar.
- **Ocupación**: estudiantes (29 %) y jubilados (23 %) convierten mucho más que blue-collar (7 %).
- **Préstamos**: tener hipoteca o préstamo personal reduce la conversión casi a la mitad.
- **contact = unknown**: solo 4 % de conversión (ver sección 5).

## 5. `unknown` como categoría con significado propio

In [ ]:
table('eda_unknown_analysis.csv')
show('eda_unknown_by_file_position.png')
table('eda_unknown_by_file_position.csv')

`unknown` **no es un faltante aleatorio**:

- `contact = unknown` ocupa el 100 % del primer 20 % del archivo y desaparece después del 30 %: corresponde a un período (mayo–junio 2008) en que no se registraba el canal. Su baja conversión (4 %) refleja ese período, no el canal.
- `poutcome = unknown` = cliente nunca contactado en una campaña previa (ver sección 6).
- La tasa de conversión crece a lo largo del archivo (de 2,9 % a 47,5 %): **hay drift temporal** fuerte en los datos. Con una partición aleatoria esto no afecta la evaluación, pero es una limitación para producción.

Decisión: mantener `unknown` como categoría propia (no imputar a la moda).

## 6. `pdays = -1`, `previous` y `poutcome`

In [ ]:
table('eda_pdays_previous_poutcome.csv')
show('eda_pdays_poutcome.png')

Las tres variables codifican lo mismo cuando el cliente nunca fue contactado: las 33.249 filas con `pdays = -1` tienen `previous = 0` y `poutcome = unknown`. Solo 5 filas tienen contacto previo pero `poutcome = unknown` (inconsistencia menor, se conservan). El -1 **no es un número**: tratarlo como tal le diría a un modelo lineal que "-1 días" está cerca de "0 días". Por eso en la Fase 3 se crea la bandera `previously_contacted` y `pdays_clean` (NaN cuando -1).

## 7. `month`: estacionalidad aparente vs volumen

In [ ]:
show('eda_month_volume_rate.png')
table('eda_month_volume_rate.csv')

Mayo concentra el 30 % de los contactos con la menor tasa (6,7 %), mientras que marzo, septiembre, octubre y diciembre tienen tasas del 43–53 % con muy poco volumen (1–2 % cada uno). La correlación de Spearman entre volumen y tasa es −0,84 (p < 0,001).

Interpretación: no es estacionalidad del cliente sino de **la operación**: los meses con campañas masivas tienen tasas bajas y los meses con pocas llamadas (dirigidas, o del final del período 2010) tienen tasas altas. Como el dataset no trae el año, `month` actúa en parte como proxy del período. Se usa como variable (se conoce antes de llamar), pero se advierte que su efecto puede no trasladarse a campañas futuras.

## 8. Outliers: `balance` y `campaign`

In [ ]:
show('eda_outliers_balance_campaign.png')
print(json.dumps(key['outliers'], indent=2))

- `balance`: 10,4 % de outliers por la regla IQR, valores entre −8.019 y 102.127. No son errores (saldos altos existen), por eso **no se eliminan**: se aplica transformación logarítmica con signo y se agregan banderas de saldo negativo (8,4 %) y cero (7,8 %).
- `campaign`: máximo 63 contactos, p99 = 17. La conversión cae de 14,7 % (1 contacto) a 6,4 % (≥ 5). Se **winsoriza al p99** (calculado dentro de cada fold) para que valores extremos no dominen los modelos lineales.

## 9. `duration`: por qué es leakage

In [ ]:
show('eda_duration_leakage.png')
print(json.dumps(key['duration'], indent=2))

`duration` por sí sola tiene ROC-AUC ≈ 0,81: más que muchos modelos completos sin ella. La mediana de duración es 164 s en los que no convierten y 424 s en los que convierten; las llamadas de menos de 60 s casi nunca convierten (0,2 %). Solo hay 3 llamadas con duración 0.

El problema: **la duración se conoce recién al terminar la llamada**, y una llamada larga es en buena medida *consecuencia* del interés del cliente. Un modelo que la use no puede decidir a quién llamar. Se excluye del modelo entregable y se usa solo para cuantificar el efecto (Fase 5).

## 10. Ranking preliminar de variables

In [ ]:
show('eda_feature_ranking.png')
table('eda_feature_ranking.csv')

Sin contar `duration`, las variables más informativas son `poutcome`, `pdays`, `month`, `balance`, `previous`, `contact`, `age` y `housing`. `default` es prácticamente no informativa (V ≈ 0,02). Nota: la V de Cramér de `pdays` es baja porque el 82 % de sus valores es −1 y al discretizar en deciles queda casi en un único grupo; la información mutua sí la detecta.

In [ ]:
show('eda_numeric_correlation.png')

Correlaciones bajas entre numéricas, salvo `pdays`–`previous` (misma información de contacto previo). No hay multicolinealidad severa que obligue a descartar variables.